# 结构化生成

在返回数据，请求数据，存储数据的时候都尽量结构化数据，如 {"departure": "上海", "destination": "北京", "date": "2025-07-18"}。这有助于更加有效明确的检索数据和生成

## Ooutput Parsers

Langchain提供了一个组件叫 OutputParsers，他专门用于处理LLM的输出，在提示词prompt里面注入指令，用来让LLM返回结构化的数据

其中具体有三种
1. StrOutputParser：最基础的输出解析器，它简单地将 LLM 的输出作为字符串返回。
2. JsonOutputParser：可以解析包含嵌套结构和列表的复杂 JSON 字符串。
3. PydanticOutputParser：通过与 Pydantic 模型结合，可以实现对输出格式最严格的定义和验证。

这里输出集成的关键就是 ： 

1. PromptTemplate 修改prompt 用来指定llm输出的结果格式
2. llm 处理prompt
3. PydanticOutputParser 来把字符进行反序列化

In [1]:
import os
from typing import List
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

# 定义pydantic中的数据类型
class PersonInfo(BaseModel):
    """用于存储个人信息的数据结构"""
    name : str = Field(description="人物姓名")
    age : int = Field(description="人物年龄")
    skills : List[str] = Field(description="技能列表")
    
# 规定下输出的格式, 这里进行反序列化，把字符串转化成真正的python数据格式
parser = PydanticOutputParser(pydantic_object=PersonInfo)

# 创建提示词模版，注入格式指令
prompt = PromptTemplate(
    template="请根据以下文本提取信息。\n{format_instructions}\n{text}\n",
    # 需要改造的信号
    input_variables=["text"], 
    
    # 这里把解析器的格式输入到大模型， 强制他输出json
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

load_dotenv()

llm = ChatOpenAI(
    temperature=0,
    model = os.getenv("AIHUBMIX_MODEL"),
    base_url=os.getenv("AIHUBMIX_BASE_URL"),
    api_key=os.getenv("AIHUBMIX_API_KEY")
)

# 这里langchain把 | 符号进行重载成管道，表示左边的结果是右边的输入
chain = prompt | llm | parser

text = "张三今年30岁，他擅长Python和Go语言。"

result = chain.invoke({"text" : text})

print(result)

name='张三' age=30 skills=['Python', 'Go语言']
